# 00 — One-time setup (run on the conda **base** kernel)
1. Creates the **`dr-torch`** kernel (Python 3.12, PyTorch CUDA 12.8 — supports the RTX 5070 Ti) used by notebooks 02–04.
2. Unzips the Roboflow export and re-creates the 75/10/15 stratified split at 224×224 with exactly the logic of `notebooks/01_preprocessing.ipynb` (seed 42) — the same split the Attention U-Net was trained/tested on.

The Attention U-Net (`notebooks/02_train_class_specific_aug.ipynb`) is the original run with its outputs; it is not re-run.

In [ ]:
import os, sys, subprocess, time
CONDA = os.path.join(sys.base_prefix, "Scripts", "conda.exe")
ENVS = os.path.join(sys.base_prefix, "envs")
def run(cmd):
    print(">", " ".join(cmd), flush=True); t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:]); print(f"[{time.time()-t:.0f}s] exit {r.returncode}", flush=True)
    assert r.returncode == 0, "command failed"
print(CONDA, os.path.exists(CONDA))

In [ ]:
# --- dr-torch : PyTorch with CUDA 12.8 (Blackwell / RTX 50-series support)
py = os.path.join(ENVS, "dr-torch", "python.exe")
if not os.path.exists(py):
    run([CONDA, "create", "-y", "-q", "-p", os.path.join(ENVS, "dr-torch"), "python=3.12"])
run([py, "-m", "pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu128"])
run([py, "-m", "pip", "install", "-q", "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "seaborn", "pillow", "ipykernel", "jinja2"])
run([py, "-m", "ipykernel", "install", "--user", "--name", "dr-torch", "--display-name", "dr-torch (PyTorch CUDA)"])
run([py, "-c", "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0)); print((torch.randn(512,512,device='cuda')@torch.randn(512,512,device='cuda')).sum().item())"])

## Data: unzip + the preprocessing notebook's split (stratified 75/10/15, seed 42, resize 224×224)

In [ ]:
import zipfile, shutil
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor

DP = r"C:\Users\pooja\Desktop\dp"
ZIP = os.path.join(DP, "Diabetic Retenopathy.v8i.folder.zip")
SOURCE_DATASET_PATH = os.path.join(DP, "Diabetic Retenopathy.v8i.folder")
OUTPUT_DATASET_PATH = os.path.join(DP, "Diabetic Retenopathy.v8i.folder.new_updated")
TARGET_SIZE, TEST_SIZE_PERCENTAGE, VALIDATION_SIZE_PERCENTAGE = (224, 224), 0.15, 0.10

with zipfile.ZipFile(ZIP) as z:
    z.extractall(SOURCE_DATASET_PATH)

rows = []
for split in ["train", "test", "validation", "valid"]:
    sp = os.path.join(SOURCE_DATASET_PATH, split)
    if not os.path.exists(sp): continue
    for c in os.listdir(sp):
        if os.path.isdir(os.path.join(sp, c)):
            for f in os.listdir(os.path.join(sp, c)):
                if f.lower().endswith((".png", ".jpg", ".jpeg")):
                    rows.append({"path": os.path.join(sp, c, f), "class": c, "original_split": split})
all_images_df = pd.DataFrame(rows)
print(len(all_images_df), "images"); print(all_images_df["class"].value_counts())

train_val_df, test_df = train_test_split(all_images_df, test_size=TEST_SIZE_PERCENTAGE, stratify=all_images_df["class"], random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=VALIDATION_SIZE_PERCENTAGE / (1 - TEST_SIZE_PERCENTAGE),
                                    stratify=train_val_df["class"], random_state=42)
print("train", len(train_df), "val", len(val_df), "test", len(test_df))

In [ ]:
def process(args):
    path, split, cls = args
    out_dir = os.path.join(OUTPUT_DATASET_PATH, split, cls); os.makedirs(out_dir, exist_ok=True)
    Image.open(path).resize(TARGET_SIZE, Image.LANCZOS).save(os.path.join(out_dir, os.path.basename(path)), quality=95)

if os.path.exists(OUTPUT_DATASET_PATH):
    shutil.rmtree(OUTPUT_DATASET_PATH)
jobs = [(r.path, s, r["class"]) for s, d in [("train", train_df), ("validation", val_df), ("test", test_df)] for _, r in d.iterrows()]
t = time.time()
with ThreadPoolExecutor(12) as ex:
    list(ex.map(process, jobs))
print(f"resized {len(jobs)} images in {time.time()-t:.0f}s")
for s in ["train", "validation", "test"]:
    print(s, {c: len(os.listdir(os.path.join(OUTPUT_DATASET_PATH, s, c))) for c in sorted(os.listdir(os.path.join(OUTPUT_DATASET_PATH, s)))})